In [2]:
using GLPK, JuMP, Plots

model = Model(GLPK.Optimizer)

alpha = 1.0; beta = 0.0   # only cost
alpha = 0.5; beta = 0.5   # balanced
alpha = 0.2; beta = 0.8   # CO2-focused

T = 10      # Time
N = 20     # Anzahl Zeiteinheiten
h = T/N   # Zeiteinheiten

# 1 = Wärmepumpe        heat pump
# 2 = Biomassekessel    biomass boiler
# 3 = Gaskessel         gas boiler
# 4 = Speichertank      thermal storage discharge

# je Heizquelle
c = [25, 35, 80, 5]         # COST              - €/MWh
s = [30, 40, 200, 5]        # SUSTAINABILITY    - kg_CO2/MWh

m = [120, 150, 100, 60]     # max. MW
# je Zeitpunkt      
"""Demand darf 430 nicht überschreiten"""
d = [430, 200, 230, 270, 300, 280, 240, 210, 420, 190, 180, 200, 0, 270, 300, 280, 240, 210, 260, 190]   # MW 
# Steigung - Steuerungsspontanität
k = [60,30,100,0]

alpha = [1,0,0.5,0.2]
M=length(alpha)

objValue = zeros(M)
# beta = 1 - alpha
for k=1:M
    @variable(model, x[1:4,1:N] >= 0)

    # KOSTEN minimieren
    # beta = (1-alpha[k])
    @objective( model, Min, alpha[k]     *  sum( sum(x[j,i]*c[j] for i=1:N) for j=1:4) + 
                            (1-alpha[k]) *  sum( sum(x[j,i]*s[j] for i=1:N) for j=1:4) )

    # CONSTRAINTS:
    # kann maximalen Wirkungsgrad nicht übersteigen    -    je Heizquelle
    @constraint(model, [j=1:4,i=1:N], x[j,i] <= m[j])
    # DEMAND muss erfüllt sein      -    für alle Zeitpunkte i
    @constraint(model, [i=1:N],   sum( x[j,i] for j=1:4) >= d[i]    )

    # Steuerungsspontanität
    # increase
    @constraint(model, [j=1:4, i=2:N],
        x[j,i] - x[j,i-1] <= k[j])
    # decrease
    @constraint(model, [j=1:4, i=2:N],
        x[j,i-1] - x[j,i] <= k[j])

    optimize!(model)

    objValue[k] = objective_value(model)
    # println("Solver status: ", termination_status(model))
    # println("Solution status: ", primal_status(model))
    ## Printing the solution
    # println("Objective value = ", objective_value(model))
    
    for j=1:4
        for i=1:N
            print(value(x[j,i]),"\t")
        end
        println(".")
    end

    cost = 0
    CO2 = 0

    for j=1:4
        for i=1:N
            cost += value(x[j,i])*c[j]
            CO2 += value(x[j,i])*s[j]
        end
    end
end
cost_min = objValue[1]
co2_min = objValue[2]

println("Cost in €: ",cost)
println("CO2 in Kg: ", CO2)

BoundsError: BoundsError: attempt to access Int64 at index [2]

In [ ]:
p = plot( 1:N, [value.(x[j,:]) for j in 1:4], labels=["Heat pump" "Biomass" "Gas" "TES"], xlabel="Time step", ylabel="Power [MW]", lw=2 , title="MW per Heatsource")
plot!(p, 1:N, d, label="Demand", lw=3, linestyle=:dash )
display(p)

x_result = value.(x)

hp  = x_result[1,:]
bio = hp .+ x_result[2,:]
gas = bio .+ x_result[3,:]
tes = gas .+ x_result[4,:]

p1 = plot(1:N, hp, label="Heat pump", lw=2,title="MW in total")
plot!(p1, 1:N, bio, label="+ Biomass", lw=2)
plot!(p1, 1:N, gas, label="+ Gas", lw=2)
plot!(p1, 1:N, tes, label="+ TES", lw=2)
plot!(p1, 1:N, d, label="Demand", lw=3, linestyle=:dash)

xlabel!(p1, "Time step")
ylabel!(p1, "Power [MW]")

display(p1)
